In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
from scipy.stats import norm as scipy_norm, invgamma
from scipy.stats import beta as scipy_beta
from scipy.stats import halfnorm
import matplotlib.pyplot as plt
import bayesflow as bf
import keras

np.random.seed(123)
np.set_printoptions(suppress=True)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Helper functions
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
from scipy.spatial import Delaunay

def _random_adjacency(N):
    """
    Generates a realistic spatial adjacency matrix A 
    using Delaunay Triangulation (planar graph).
    """
    # 1. Drop N random coordinates in a 2D space
    points = np.random.uniform(0, 10, size=(N, 2))
    
    # 2. Compute Delaunay Triangulation (creates non-overlapping triangles)
    tri = Delaunay(points)
    
    # 3. Build the Adjacency Matrix A from the triangle edges
    A = np.zeros((N, N), dtype=int)
    for simplex in tri.simplices:
        # simplex contains the indices of the 3 points forming a triangle
        for i in range(3):
            for j in range(i + 1, 3):
                n1, n2 = simplex[i], simplex[j]
                A[n1, n2] = 1
                A[n2, n1] = 1  # Make it symmetric
                
    return A

def _dagar_factors(A, rho, ordering):
    """
    Compute the DAGAR factors (ImB, lam) without forming Q explicitly.
 
    Q(rho) = (I-B)^T diag(lam) (I-B)
 
    Returns
    -------
    ImB : (n, n)  unit lower-triangular matrix (I - B)
    lam : (n,)    diagonal entries of Lambda
 
    Sampling w ~ N(0, sigma2_w * Q^{-1}) is then done via triangular solve:
        z   ~ N(0, I)
        rhs = sqrt(sigma2_w) * z / sqrt(lam)     [scale by Lambda^{-1/2}]
        w   = solve(ImB, rhs)                     [O(N^2) back-substitution]
    which avoids forming Q or its inverse entirely.
    """
    n    = A.shape[0]
    rho2 = rho ** 2
    inv_order = np.argsort(ordering)
 
    B   = np.zeros((n, n))
    lam = np.zeros(n)
 
    for pos in range(n):
        i     = ordering[pos]
        preds = [ordering[q] for q in range(pos) if A[i, ordering[q]] == 1]
        n_lt  = len(preds)
        denom = 1.0 + max(n_lt - 1, 0) * rho2
        b_val = rho / denom if n_lt > 0 else 0.0
        for j in preds:
            B[pos, inv_order[j]] = b_val
        lam[pos] = denom / (1.0 - rho2)
 
    ImB = np.eye(n) - B   # unit lower-triangular
    return ImB, lam

In [ ]:
import warnings

# con rho ed N fissati funziona

ETA_RAW_LOWER = 0.0

# Toggle this flag only for N
USE_FIXED_N = False

ORDERING_MODE = "identity"   # "identity", "random", "x_sorted"

# Used when USE_FIXED_N = True
N_FIXED = 100

# Used when USE_FIXED_N = False
N_MIN = 40
N_MAX = 300

# If you already loaded a fixed adjacency matrix A earlier in the notebook,
# the simulator will use it. Otherwise it will generate a random adjacency.
if "A" in globals():
    A_GLOBAL = np.asarray(A, dtype=np.float32)
    N_FIXED = int(A_GLOBAL.shape[0])
else:
    A_GLOBAL = None
    N_FIXED = 100


def _get_simulation_adjacency(N):
    if A_GLOBAL is not None:
        if A_GLOBAL.shape[0] != N:
            raise ValueError(f"A has shape {A_GLOBAL.shape}, but N={N}.")
        return A_GLOBAL.copy()
    return _random_adjacency(N).astype(np.float32)

"""
def _deterministic_ordering(A):
    # Current recommendation: fixed identity ordering.
    # This matched the clearest eta signal in your diagnostics.
    return np.arange(A.shape[0], dtype=int)
"""

def _get_ordering(A):
    n = A.shape[0]

    if ORDERING_MODE == "random":
        return np.random.permutation(n).astype(int)
    if ORDERING_MODE == "identity":
        return np.arange(n, dtype=int)

    raise ValueError(f"Unknown ORDERING_MODE: {ORDERING_MODE}")

def _repair_isolates_deterministic(A_filtered, A, Z):
    A_rep = A_filtered.copy().astype(np.float32)
    for i in range(A_rep.shape[0]):
        if A_rep[i].sum() == 0:
            neighbors = np.where(A[i] == 1)[0]
            if len(neighbors) > 0:
                # Reconnect to the most similar observed neighbor
                j = neighbors[np.argmin(Z[i, neighbors])]
                A_rep[i, j] = 1.0
                A_rep[j, i] = 1.0
    return A_rep


def _masked_row_mean(values, mask):
    mask_f = mask.astype(np.float32)
    denom = mask_f.sum(axis=1)
    denom_safe = np.where(denom == 0, 1.0, denom)
    out = (values * mask_f).sum(axis=1) / denom_safe
    out[denom == 0] = 0.0
    return out.astype(np.float32)


def _safe_mean_1d(values):
    return float(values.mean()) if values.size > 0 else 0.0

In [ ]:
def _safe_corr(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    x_c = x - x.mean()
    y_c = y - y.mean()

    denom = np.sqrt(np.mean(x_c ** 2) * np.mean(y_c ** 2))
    if denom < 1e-8:
        return 0.0
    return float(np.mean(x_c * y_c) / denom)


def _observed_eta_signal_metrics(y, e, A, Z, Z_median, r_lag_all=None):
    A_bool = A == 1
    edge_i, edge_j = np.where(np.triu(A_bool, 1))

    zero_metrics = dict(
        edge_absdiff_low=0.0,
        edge_absdiff_mid=0.0,
        edge_absdiff_high=0.0,
        edge_concord_low=0.0,
        edge_concord_mid=0.0,
        edge_concord_high=0.0,
        edge_absdiff_slope=0.0,
        edge_absdiff_gap=0.0,
        edge_concord_gap=0.0,
        edge_corr_all=0.0,
        lag_corr_all=0.0,
        lag_slope_all=0.0,
        edge_semivar_all=0.0,
        local_moran_mean=0.0,
    )
    if edge_i.size == 0:
        return zero_metrics

    r = (np.log(y + 0.5) - np.log(e)).astype(np.float32)
    z_rel = (Z / Z_median).astype(np.float32)

    z_edge = z_rel[edge_i, edge_j]
    absdiff_edge = np.abs(r[edge_i] - r[edge_j]).astype(np.float32)
    sqdiff_edge = ((r[edge_i] - r[edge_j]) ** 2).astype(np.float32)

    r_centered = (r - r.mean()).astype(np.float32)
    var_r = float(np.mean(r_centered ** 2))
    var_r_safe = max(var_r, 1e-8)

    concord_edge = (r_centered[edge_i] * r_centered[edge_j]).astype(np.float32)

    low = z_edge <= 0.75
    mid = (z_edge > 0.75) & (z_edge <= 1.25)
    high = z_edge > 1.25

    z_bar = float(z_edge.mean())
    absdiff_bar = float(absdiff_edge.mean())
    var_z = float(np.mean((z_edge - z_bar) ** 2))

    if var_z < 1e-8:
        edge_absdiff_slope = 0.0
    else:
        edge_absdiff_slope = float(
            np.mean((z_edge - z_bar) * (absdiff_edge - absdiff_bar)) / var_z
        )

    if r_lag_all is None:
        degree = A_bool.sum(axis=1).astype(np.float32)
        degree_safe = np.where(degree == 0, 1.0, degree)
        W = A / degree_safe[:, None]
        r_lag_all = (W @ r).astype(np.float32)

    r_lag_centered = (r_lag_all - r_lag_all.mean()).astype(np.float32)
    lag_slope_all = float(np.mean(r_centered * r_lag_centered) / var_r_safe)
    lag_corr_all = _safe_corr(r, r_lag_all)

    local_moran = (r_centered * r_lag_all / var_r_safe).astype(np.float32)
    edge_corr_all = float(np.mean(concord_edge) / var_r_safe)
    edge_semivar_all = float(0.5 * np.mean(sqdiff_edge))

    return dict(
        edge_absdiff_low=_safe_mean_1d(absdiff_edge[low]),
        edge_absdiff_mid=_safe_mean_1d(absdiff_edge[mid]),
        edge_absdiff_high=_safe_mean_1d(absdiff_edge[high]),
        edge_concord_low=_safe_mean_1d(concord_edge[low]),
        edge_concord_mid=_safe_mean_1d(concord_edge[mid]),
        edge_concord_high=_safe_mean_1d(concord_edge[high]),
        edge_absdiff_slope=edge_absdiff_slope,
        edge_absdiff_gap=_safe_mean_1d(absdiff_edge[high]) - _safe_mean_1d(absdiff_edge[low]),
        edge_concord_gap=_safe_mean_1d(concord_edge[low]) - _safe_mean_1d(concord_edge[high]),
        edge_corr_all=edge_corr_all,
        lag_corr_all=lag_corr_all,
        lag_slope_all=lag_slope_all,
        edge_semivar_all=edge_semivar_all,
        local_moran_mean=float(local_moran.mean()),
    )


def _build_observed_features(x, y, e, A, Z, Z_median, M):
    A_bool = A == 1

    x = x.astype(np.float32)
    y = y.astype(np.float32)
    e = e.astype(np.float32)

    log_y = np.log1p(y).astype(np.float32)
    log_e = np.log(e).astype(np.float32)
    r = (np.log(y + 0.5) - log_e).astype(np.float32)

    r_centered = (r - r.mean()).astype(np.float32)
    var_r_safe = max(float(np.mean(r_centered ** 2)), 1e-8)

    degree = A_bool.sum(axis=1).astype(np.float32)
    degree_safe = np.where(degree == 0, 1.0, degree)

    z_rel = (Z / Z_median).astype(np.float32)
    neigh_r = np.broadcast_to(r[None, :], z_rel.shape).astype(np.float32)
    abs_r_diff = np.abs(r[:, None] - r[None, :]).astype(np.float32)

    low_mask = A_bool & (z_rel <= 0.75)
    mid_mask = A_bool & (z_rel > 0.75) & (z_rel <= 1.25)
    high_mask = A_bool & (z_rel > 1.25)

    r_lag_all = _masked_row_mean(neigh_r, A_bool)
    absdiff_all = _masked_row_mean(abs_r_diff, A_bool)

    r_lag_low = _masked_row_mean(neigh_r, low_mask)
    r_lag_mid = _masked_row_mean(neigh_r, mid_mask)
    r_lag_high = _masked_row_mean(neigh_r, high_mask)

    absdiff_low = _masked_row_mean(abs_r_diff, low_mask)
    absdiff_mid = _masked_row_mean(abs_r_diff, mid_mask)
    absdiff_high = _masked_row_mean(abs_r_diff, high_mask)

    prop_low = (low_mask.sum(axis=1) / degree_safe).astype(np.float32)
    prop_mid = (mid_mask.sum(axis=1) / degree_safe).astype(np.float32)
    prop_high = (high_mask.sum(axis=1) / degree_safe).astype(np.float32)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_z_rel = np.nanmean(np.where(A_bool, z_rel, np.nan), axis=1)
        max_z_rel = np.nanmax(np.where(A_bool, z_rel, np.nan), axis=1)

    mean_z_rel = np.nan_to_num(mean_z_rel, nan=0.0).astype(np.float32)
    max_z_rel = np.nan_to_num(max_z_rel, nan=0.0).astype(np.float32)

    edge_metrics = _observed_eta_signal_metrics(
        y=y,
        e=e,
        A=A,
        Z=Z,
        Z_median=Z_median,
        r_lag_all=r_lag_all,
    )

    local_moran = (r_centered * r_lag_all / var_r_safe).astype(np.float32)
    local_semivar = (((r - r_lag_all) ** 2) / var_r_safe).astype(np.float32)

    obs = np.stack(
        [
            x,
            #y,
            #e,
            log_y,
            log_e,
            r,
            degree,
            r_lag_all,
            absdiff_all,
            r_lag_low,
            r_lag_mid,
            r_lag_high,
            absdiff_low,
            absdiff_mid,
            absdiff_high,
            prop_low,
            prop_mid,
            prop_high,
            mean_z_rel,
            max_z_rel,
            local_moran,
            local_semivar,
            np.full(len(x), M, dtype=np.float32),
            np.full(len(x), edge_metrics["edge_absdiff_slope"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_absdiff_gap"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_concord_gap"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_corr_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["lag_corr_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["lag_slope_all"], dtype=np.float32),
            np.full(len(x), edge_metrics["edge_semivar_all"], dtype=np.float32),
        ],
        axis=-1,
    )

    return obs.astype(np.float32)


In [ ]:
def prior():
    beta_0 = np.random.normal(0.0, 0.5)
    sigma2_w = halfnorm.rvs(scale=0.5)
    eta_raw = np.random.uniform(ETA_RAW_LOWER, 1.0)
    rho = np.random.uniform(0.0, 1.0)
    return dict(
        beta=np.array([beta_0], dtype=np.float32),
        sigma2_w=np.array([sigma2_w], dtype=np.float32),
        eta_raw=np.array([eta_raw], dtype=np.float32),
        rho=np.array([rho], dtype=np.float32),
    )


def likelihood(beta, sigma2_w, eta_raw, rho, N):
    beta_0 = float(beta[0])
    sigma2_w = float(sigma2_w[0])
    eta_raw_val = float(eta_raw[0])
    rho_val = float(rho[0])
    N = int(N)

    A_sim = _get_simulation_adjacency(int(N)).astype(np.float32)

    x = np.random.normal(0.0, 1.0, size=N).astype(np.float32)
    Z = np.abs(x[:, None] - x[None, :]).astype(np.float32)

    Z_edges = Z[A_sim == 1]
    if Z_edges.size == 0:
        Z_median = 1.0
    else:
        Z_median = float(np.median(Z_edges) + 1e-8)

    M = float(np.log(2.0) / Z_median)
    eta = float(eta_raw_val * M)

    A_filtered = A_sim * ((Z * eta) <= np.log(2.0)).astype(np.float32)
    A_filtered = _repair_isolates_deterministic(A_filtered, A_sim, Z)

    ordering = _get_ordering(A_sim)
    
    ImB, dagar_lam = _dagar_factors(A_filtered, rho_val, ordering) #RHO_VAL
    z = np.random.normal(size=N).astype(np.float32)
    rhs = np.sqrt(sigma2_w) * z / np.sqrt(dagar_lam)
    w = np.linalg.solve(ImB, rhs).astype(np.float32)
    w = w - np.mean(w)

    # e = np.random.uniform(50.0, 150.0, size=N).astype(np.float32)
    log_e = np.random.uniform(np.log(2.0), np.log(30000.0), size=N)
    e = np.exp(log_e).astype(np.float32)

    log_poisson_lam = np.log(e) + beta_0 + w
    poisson_lam = np.clip(np.exp(log_poisson_lam), 1e-2, 1e6)
    y = np.random.poisson(poisson_lam).astype(np.float32)

    obs = _build_observed_features(
        x=x,
        y=y,
        e=e,
        A=A_sim,
        Z=Z,
        Z_median=Z_median,
        M=M,
    )

    return dict(
        obs=obs,
        M=np.array([M], dtype=np.float32),
        A=A_sim.astype(np.float32),
        A_filtered=A_filtered.astype(np.float32),
    )


def meta():
    if USE_FIXED_N:
        return dict(N=int(N_FIXED))
    return dict(N=int(np.random.randint(N_MIN, N_MAX + 1)))

In [ ]:
simulator = bf.simulators.make_simulator([prior, likelihood], meta_fn=meta)

# Sanity check
sim_draws = simulator.sample(32)
print("Simulator sanity check:")
print("  N             :", sim_draws["N"])
print("  beta     shape:", sim_draws["beta"].shape)
print("  sigma2_w shape:", sim_draws["sigma2_w"].shape)
print("  eta_raw  shape:", sim_draws["eta_raw"].shape)
print("  obs      shape:", sim_draws["obs"].shape)
print("  M        shape:", sim_draws["M"].shape)
print("  A        shape:", sim_draws["A"].shape)
print("  A_filt   shape:", sim_draws["A_filtered"].shape)
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Adapter
# ─────────────────────────────────────────────────────────────────────────────

adapter = (
    bf.Adapter()
    .broadcast("N", to="obs")
    .constrain("sigma2_w", lower=0, method="softplus")
    .constrain("eta_raw", lower=0, upper=1, method="sigmoid")
    .constrain("rho", lower=0, upper=1, method="sigmoid")
    .convert_dtype("float64", "float32")
    .concatenate(["beta", "sigma2_w", "eta_raw", "rho"], into="inference_variables")
    .rename("obs", "summary_variables")
    .keep(["inference_variables", "summary_variables", "M", "A", "A_filtered"])
)

processed = adapter(sim_draws)
print("Processed shapes:")
print("  inference_variables :", processed["inference_variables"].shape)
print("  summary_variables   :", processed["summary_variables"].shape)
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Networks
# ─────────────────────────────────────────────────────────────────────────────

summary_network = bf.networks.SetTransformer(summary_dim=32) #64

inference_network = bf.networks.CouplingFlow(transform="spline") # num_layers=6, hidden_units=128,

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Workflow
# ─────────────────────────────────────────────────────────────────────────────

workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_network,
    summary_network=summary_network,
    standardize=["inference_variables","summary_variables"]
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Training 
# ─────────────────────────────────────────────────────────────────────────────

print("Training ...")
history = workflow.fit_online(epochs=100, batch_size=64, num_batches_per_epoch=200) 
print("Training complete.\n")

In [ ]:
import pandas as pd
from pathlib import Path

def find_repository_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Training").is_dir() and (candidate / "Simulation Experiments").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ABI_poisson_regression repository root.")


REPOSITORY_ROOT = find_repository_root()

RESULTS_DIR = REPOSITORY_ROOT / "Training"
CHECKPOINTS_DIR = RESULTS_DIR / "Checkpoints"
IMAGES_DIR = RESULTS_DIR / "Images"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Save model weights
workflow.approximator.save(str(CHECKPOINTS_DIR / "poisson_dagar.keras"))

# Save loss history
pd.DataFrame(history.history).to_csv(RESULTS_DIR / "training_history.csv", index=False)

print("Saved.")

In [ ]:
# Reload model
workflow = keras.saving.load_model(str(CHECKPOINTS_DIR / "poisson_dagar.keras"))

# Reload history for loss plot
hist_df = pd.read_csv(RESULTS_DIR / "training_history.csv")

# Reconstruct history-like object for bf.diagnostics.plots.loss()
class FakeHistory:
    def __init__(self, df):
        self.history = df.to_dict(orient="list")

history = FakeHistory(hist_df)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Diagnostics
# ─────────────────────────────────────────────────────────────────────────────

par_names = [r"$\beta_0$", r"$\sigma^2_w$", r"$\eta_{raw}$", r"$\eta$", r"$\rho$"]#, "mean_deg", "mean_lag", "mean_diff"

# Loss curve
f = bf.diagnostics.plots.loss(history)

plt.savefig(IMAGES_DIR/"history_plot.png",
            dpi=150, bbox_inches="tight")
print("History plot saved.")